# ML Leakage Detector Demo

This notebook demonstrates how the `ml-leakage-detector` tool identifies various data leakage patterns in Python code. It's a great way to understand the types of issues the detector can catch and how it presents its findings.

In [ ]:
from src.detector import LeakageDetector
from src.patterns import LeakagePattern
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest
from sklearn.pipeline import Pipeline

## Initialize the Detector

In [ ]:
detector = LeakageDetector()

## Scenario 1: Imputation Leakage (Critical)

In [ ]:
buggy_imputation_code = """
import pandas as pd
df = pd.DataFrame({'col1': [1, 2, np.nan, 4]})
df_filled = df.fillna(df.mean()) # Leak: mean from full dataset
X_train, X_test = train_test_split(df_filled)
"""

print("--- Testing Imputation Leak ---")
result = detector.detect(buggy_imputation_code)
print(detector.generate_report(result))

## Scenario 2: Scaling Leakage (Critical)

In [ ]:
buggy_scaling_code = """
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # Leak: fit on full dataset
X_train, X_test = train_test_split(X_scaled)
"""

print("--- Testing Scaling Leak ---")
result = detector.detect(buggy_scaling_code)
print(detector.generate_report(result))

## Scenario 3: Sequential Split (Medium)

In [ ]:
buggy_sequential_split_code = """
split_idx = int(0.8 * len(X))
X_train = X[:split_idx] # Leak: sequential slicing
X_test = X[split_idx:]
"""

print("--- Testing Sequential Split Leak ---")
result = detector.detect(buggy_sequential_split_code)
print(detector.generate_report(result))

## Scenario 4: Feature Selection Leakage (Critical)

In [ ]:
buggy_feature_selection_code = """
from sklearn.feature_selection import SelectKBest
selector = SelectKBest(k=10).fit(X, y) # Leak: fit on full dataset
X_selected = selector.transform(X)
X_train, X_test = train_test_split(X_selected)
"""

print("--- Testing Feature Selection Leak ---")
result = detector.detect(buggy_feature_selection_code)
print(detector.generate_report(result))

## Scenario 5: Time-Series Shuffle Leakage (High)

In [ ]:
buggy_time_series_shuffle_code = """
from sklearn.model_selection import KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42) # Leak: shuffling time-series data
"""

print("--- Testing Time-Series Shuffle Leak ---")
result = detector.detect(buggy_time_series_shuffle_code)
print(detector.generate_report(result))

## Scenario 6: Correct Pipeline (No Leakage)

In [ ]:
correct_code = """
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
pipeline.fit(X_train, y_train)
"""

print("--- Testing Correct Pipeline ---")
result = detector.detect(correct_code)
print(detector.generate_report(result))

## Scenario 7: Multiple Leaks in One File

In [ ]:
multiple_leaks_code = """
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.DataFrame({'col1': [1, 2, np.nan, 4]})
df_filled = df.fillna(df.mean()) # Leak 1: Imputation

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df_filled) # Leak 2: Scaling

split_idx = int(0.8 * len(X_scaled))
X_train = X_scaled[:split_idx] # Leak 3: Sequential Split
"""

print("--- Testing Multiple Leaks ---")
result = detector.detect(multiple_leaks_code)
print(detector.generate_report(result))